[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eygpcr/biyofizik2026-martini/blob/main/notebooks/04_martini_input.ipynb)

# Oturum 4 — Martini 3 Girdi Hazırlama

**Biyofizik 2026 Kursu · 25 Ağustos 2026 · Dr. Öğr. Üyesi Ekrem Yaşar**

Sabah oturumlarında CHARMM-GUI arayüzü ile atomistik olarak hazırlanan protein
(`6JOD` A zinciri, anjiyotensin II tip-2 reseptörü), bu not defterinde komut
satırı araçlarıyla ve Martini 3 kaba-taneli modeli kullanılarak yeniden
hazırlanmaktadır.

Kursta üretim simülasyonu koşulmamaktadır; amaç simülasyona girecek sistemin
eksiksiz biçimde kurulmasıdır. Oturum sonunda GROMACS ile doğrudan
çalıştırılabilecek eksiksiz bir dosya kümesi Google Drive'a kaydedilmektedir.

Her adımın sonunda üretilen yapı etkileşimli olarak görüntülenmekte ve statik
bir çizimi kaydedilmektedir.

## İçindekiler

| Bölüm | Konu |
|---|---|
| 1 | Google Drive bağlanması ve çalışma klasörü |
| 2 | Yazılım kurulumu |
| 3 | Görselleştirme araçları |
| 4 | Yapının indirilmesi ve zincirlerin incelenmesi |
| 5 | Membrana göre yönlendirme (OPM) |
| 6 | A zincirinin ayıklanması |
| 7 | İkincil yapının belirlenmesi |
| 8 | `martinize2` — elastik ağ ile kaba-taneli model |
| 9 | Yapısal kısıt modelleri: elastik ağ, Gō-Martini, OLIVES |
| 10 | Martini 3 kuvvet alanı dosyaları |
| 11 | `insane` ile membran, çözücü ve iyon eklenmesi |
| 12 | İyon adlarının düzeltilmesi |
| 13 | Topolojinin tamamlanması |
| 14 | Simülasyon parametre dosyaları (`.mdp`) |
| 15 | İndeks grupları (`index.ndx`) |
| 16 | `gmx grompp` ile doğrulama |
| 17 | Sistemin analizi ve görselleştirilmesi |
| 18 | Simülasyon paketinin Drive'a kaydedilmesi |


---
## 1. Google Drive bağlanması

Colab çalışma zamanı sonlandığında üretilen dosyalar silinmektedir. Bu nedenle
çıktılar Google Drive üzerinde kalıcı bir klasöre kaydedilecektir.

**Aşağıdaki hücre ne yapıyor?** Google Drive'ı `/content/drive` altına
bağlamakta ve klasör yapısını oluşturmaktadır. Çalıştırıldığında Google
hesabına erişim izni istenecektir; açılan pencerede kendi hesabınızı seçip izin
veriniz.

```
Drive'ım/
└── Biyofizik2026_Martini/
    └── martini_input/
        ├── girdi/           ham ve hazırlanmış yapılar
        ├── cikti/           ara çıktı dosyaları
        ├── gorseller/       çizimler
        └── simulasyon/      GROMACS ile çalıştırılabilir eksiksiz paket
```

Ağır dosya işlemleri Drive üzerinde yavaş olduğundan hesaplama yerel bir
dizinde yapılmakta, üretilen dosyalar adım adım Drive'a kopyalanmaktadır.


In [ ]:
from google.colab import drive
import os, shutil, glob

drive.mount('/content/drive')

DRIVE_KOK = '/content/drive/MyDrive/Biyofizik2026_Martini'
OTURUM    = os.path.join(DRIVE_KOK, 'martini_input')
D_GIRDI   = os.path.join(OTURUM, 'girdi')
D_CIKTI   = os.path.join(OTURUM, 'cikti')
D_GORSEL  = os.path.join(OTURUM, 'gorseller')
D_SIM     = os.path.join(OTURUM, 'simulasyon')

for d in (DRIVE_KOK, OTURUM, D_GIRDI, D_CIKTI, D_GORSEL, D_SIM):
    os.makedirs(d, exist_ok=True)

CALISMA = '/content/calisma'
os.makedirs(CALISMA, exist_ok=True)
os.chdir(CALISMA)

print('Drive klasoru :', OTURUM)
print('Calisma dizini:', os.getcwd())


**Aşağıdaki hücre ne yapıyor?** Dosyaları Drive'a kopyalayan yardımcı bir
fonksiyon tanımlamaktadır. Not defteri boyunca her adımın sonunda
çağrılacaktır; böylece oturum yarıda kesilse bile o ana kadar üretilen
dosyalar korunur.


In [ ]:
def boyut_str(bayt):
    """Dosya boyutunu uygun birimde bicimlendirir."""
    if bayt < 1024:
        return f'{bayt} B'
    if bayt < 1024**2:
        return f'{bayt/1024:.1f} KB'
    return f'{bayt/1024**2:.1f} MB'


def kaydet(desenler, hedef, sessiz=False):
    """Verilen dosya desenlerini Drive'daki hedef klasore kopyalar."""
    kopyalanan = []
    for desen in desenler:
        for dosya in glob.glob(desen):
            if os.path.isfile(dosya):
                shutil.copy(dosya, hedef)
                kopyalanan.append(os.path.basename(dosya))
    if not sessiz:
        if kopyalanan:
            print(f"Drive'a kaydedildi ({os.path.basename(hedef)}/): "
                  + ', '.join(sorted(kopyalanan)))
        else:
            print('Kopyalanacak dosya bulunamadi:', desenler)
    return kopyalanan

print('kaydet() hazir.')


---
## 2. Yazılım kurulumu

**Aşağıdaki hücre ne yapıyor?** Oturumda kullanılacak dört bileşeni
kurmaktadır. Çıktı `%%capture` ile bastırılmıştır. Çalışma süresi yaklaşık
3–5 dakikadır ve oturum başına bir kez çalıştırılması yeterlidir.

| Paket | İşlevi |
|---|---|
| `vermouth` | `martinize2` komutunu sağlamaktadır |
| `insane` | Membran ve kutu inşası |
| `py3Dmol` | Yapıların not defteri içinde görüntülenmesi |
| `gromacs` | `gmx grompp` ile doğrulama |


In [ ]:
%%capture
!pip install -q vermouth insane py3Dmol
!apt-get -qq update
!apt-get -qq install -y gromacs


**Aşağıdaki hücre ne yapıyor?** Kurulumun başarılı olduğunu doğrulamaktadır.
Her satırda bir sürüm bilgisi görmelisiniz. Hata alırsanız yukarıdaki kurulum
hücresini yeniden çalıştırınız.


In [ ]:
!martinize2 --version 2>&1 | head -2
!insane --help 2>&1 | head -3
!gmx --version 2>&1 | grep -i 'GROMACS version'
import py3Dmol; print('py3Dmol hazir')


---
## 3. Görselleştirme araçları

**Aşağıdaki hücre ne yapıyor?** Not defteri boyunca kullanılacak görselleştirme
fonksiyonlarını tanımlamaktadır. Hiçbir çıktı üretmez, yalnızca tanım yapar.

| Fonksiyon | İşlevi |
|---|---|
| `yapi_goster()` | Yapıyı **etkileşimli** gösterir (döndürülebilir, yakınlaştırılabilir) |
| `koordinat_oku()` | PDB/GRO dosyasından rezidü adı, atom adı ve koordinat okur |
| `bilesen_maskeleri()` | Protein / lipit / çözücü / iyon ayrımı yapar |
| `kesit_ciz()` | **Statik** kesit çizimi üretip PNG kaydeder |
| `z_profili()` | z ekseni boyunca bileşen dağılımını çizer |
| `yonelim_olc()` | Proteinin uzun ekseninin z ile açısını ölçer |


In [ ]:
import py3Dmol
import numpy as np
import matplotlib.pyplot as plt

LIPIT  = {'POPC','POPE','POPS','POPG','CHOL','DOPC','DPPC','DOPE'}
COZUCU = {'W','WF','PW'}
IYON   = {'NA','CL','NA+','CL-','ION','K','K+'}
RENK   = {'Protein': '#2E5FA3', 'Lipit': '#E08A2E',
          'Cozucu': '#9BC49B', 'Iyon': '#C0392B'}


def yapi_goster(dosya, stil='karton', genislik=800, yukseklik=500,
                zincir_renkleri=None, cozucu_gizle=True):
    """Yapiyi not defteri icinde etkilesimli olarak gosterir."""
    bicim = 'gro' if dosya.endswith('.gro') else 'pdb'
    v = py3Dmol.view(width=genislik, height=yukseklik)
    v.addModel(open(dosya).read(), bicim)
    if zincir_renkleri:
        v.setStyle({}, {'cartoon': {'color': 'white'}})
        for zincir, renk in zincir_renkleri.items():
            v.setStyle({'chain': zincir}, {'cartoon': {'color': renk}})
    elif stil == 'kure':
        v.setStyle({}, {'sphere': {'radius': 1.6}})
    elif stil == 'cg_protein':
        v.setStyle({}, {'sphere': {'radius': 1.8, 'color': '#7F8C8D'}})
        v.setStyle({'atom': 'BB'}, {'sphere': {'radius': 2.4, 'color': '#2E5FA3'}})
    elif stil == 'sistem':
        v.setStyle({}, {})
        if not cozucu_gizle:
            v.addStyle({'resn': list(COZUCU)},
                       {'sphere': {'radius': 0.7, 'color': '#BBD8BB', 'opacity': 0.35}})
        v.addStyle({'resn': list(LIPIT)},
                   {'sphere': {'radius': 1.2, 'color': '#E08A2E', 'opacity': 0.65}})
        v.addStyle({'resn': list(IYON)},
                   {'sphere': {'radius': 1.6, 'color': '#C0392B'}})
        v.addStyle({'resn': list(LIPIT | COZUCU | IYON), 'invert': True},
                   {'sphere': {'radius': 2.2, 'color': '#2E5FA3'}})
    else:
        v.setStyle({}, {'cartoon': {'color': 'spectrum'}})
    v.zoomTo(); v.setBackgroundColor('white')
    return v.show()


def koordinat_oku(dosya, max_atom=400_000):
    """PDB veya GRO dosyasindan rezidu adi, atom adi ve koordinat (nm) okur."""
    if dosya.endswith('.gro'):
        satirlar = open(dosya).read().splitlines()
        n = int(satirlar[1])
        atomlar = satirlar[2:2+n]
        if n > max_atom:
            atomlar = atomlar[::(n // max_atom + 1)]
        res = np.array([s[5:10].strip()  for s in atomlar])
        ad  = np.array([s[10:15].strip() for s in atomlar])
        xyz = np.array([[float(s[20:28]), float(s[28:36]), float(s[36:44])]
                        for s in atomlar])
    else:
        satirlar = [l for l in open(dosya) if l.startswith(('ATOM  ', 'HETATM'))]
        res = np.array([l[17:20].strip() for l in satirlar])
        ad  = np.array([l[12:16].strip() for l in satirlar])
        xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                        for l in satirlar]) / 10.0   # A -> nm
    return res, ad, xyz


def bilesen_maskeleri(res):
    diger = list(LIPIT | COZUCU | IYON)
    return {'Protein': ~np.isin(res, diger),
            'Lipit'  : np.isin(res, list(LIPIT)),
            'Cozucu' : np.isin(res, list(COZUCU)),
            'Iyon'   : np.isin(res, list(IYON))}


def kesit_ciz(dosya, png, baslik, dilim=None, eksen=('x', 'z')):
    """Yapinin izdusum gorunumunu cizer ve PNG olarak kaydeder."""
    res, ad, xyz = koordinat_oku(dosya)
    i = {'x': 0, 'y': 1, 'z': 2}[eksen[0]]
    j = {'x': 0, 'y': 1, 'z': 2}[eksen[1]]
    k = 3 - i - j
    sec = np.ones(len(res), dtype=bool)
    if dilim is not None:
        orta = (xyz[:, k].min() + xyz[:, k].max()) / 2
        sec = np.abs(xyz[:, k] - orta) < dilim / 2
    maskeler = bilesen_maskeleri(res)
    plt.figure(figsize=(7.5, 7.5))
    for ad_g, boyut, saydam in [('Cozucu', 1.6, 0.22), ('Lipit', 3.0, 0.60),
                                ('Iyon', 8.0, 0.85), ('Protein', 6.0, 0.95)]:
        m = maskeler[ad_g] & sec
        if m.sum() == 0:
            continue
        plt.scatter(xyz[m, i], xyz[m, j], s=boyut, c=RENK[ad_g], alpha=saydam,
                    linewidths=0,
                    label=f'{ad_g} ({m.sum():,} / {maskeler[ad_g].sum():,})')
    plt.xlabel(f'{eksen[0]} (nm)'); plt.ylabel(f'{eksen[1]} (nm)')
    plt.title(baslik); plt.gca().set_aspect('equal')
    plt.legend(loc='upper right', framealpha=.9, markerscale=4,
               title='dilimde / toplam', title_fontsize=8, fontsize=8)
    plt.grid(alpha=.2); plt.tight_layout()
    plt.savefig(png, dpi=150); plt.show()
    print('Kaydedildi:', png)


def z_profili(dosya, png, baslik):
    res, ad, xyz = koordinat_oku(dosya)
    z = xyz[:, 2]
    maskeler = bilesen_maskeleri(res)
    kenar = np.linspace(z.min(), z.max(), 120)
    plt.figure(figsize=(9, 4.5))
    for ad_g, m in maskeler.items():
        if m.sum() == 0:
            continue
        plt.hist(z[m], bins=kenar, histtype='step', lw=1.7,
                 color=RENK[ad_g], label=f'{ad_g} (n={m.sum():,})')
    plt.xlabel('z (nm)'); plt.ylabel('Parcacik sayisi')
    plt.title(baslik); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(png, dpi=150); plt.show()
    print('Kaydedildi:', png)


def yonelim_olc(dosya, etiket=''):
    """Proteinin en uzun asal ekseninin z ekseni ile acisini olcer."""
    res, ad, xyz = koordinat_oku(dosya)
    m = bilesen_maskeleri(res)['Protein']
    c = xyz[m] - xyz[m].mean(0)
    _, _, vt = np.linalg.svd(c, full_matrices=False)
    aci = np.degrees(np.arccos(abs(vt[0][2])))
    boyut = c.max(0) - c.min(0)
    print(f'{etiket}')
    print(f'  en uzun eksenin z ile acisi : {aci:5.1f} derece')
    print(f'  uzanim x, y, z (nm)         : {boyut[0]:.1f}, {boyut[1]:.1f}, {boyut[2]:.1f}')
    print(f'  z / xy orani                : {boyut[2]/max(boyut[0], boyut[1]):.2f}')
    return aci

print('Gorsellestirme fonksiyonlari hazir.')


---
## 4. Yapının indirilmesi ve zincirlerin incelenmesi

**Aşağıdaki hücre ne yapıyor?** `6JOD` yapısını RCSB Protein Veri Bankası'ndan
indirmektedir. Bu dosya kristal çerçevesindeki ham koordinatları içermektedir.


In [ ]:
!wget -q https://files.rcsb.org/download/6JOD.pdb -O 6jod_rcsb.pdb
!ls -lh 6jod_rcsb.pdb


**Aşağıdaki hücre ne yapıyor?** Yapıyı zincirlere göre renklendirerek
etkileşimli olarak göstermektedir. Görünümü fare ile döndürebilirsiniz.

| Renk | Zincir | Tanım | İşlem |
|---|---|---|---|
| Kırmızı | A | Anjiyotensin II tip-2 reseptörü (AT2R) | **korunur** |
| Sarı | B | Anjiyotensin II — agonist peptit | korunur |
| Gri | C | BRIL (çözünür sitokrom b562) — kristalizasyon füzyonu | **çıkarılır** |
| Açık mavi | H | 4A03 Fab ağır zincir | **çıkarılır** |
| Açık yeşil | L | 4A03 Fab hafif zincir | **çıkarılır** |

Yapının ne kadar büyük bir bölümünün deneysel yardımcı bileşenlerden oluştuğuna
dikkat ediniz.


In [ ]:
yapi_goster('6jod_rcsb.pdb', zincir_renkleri={
    'A': 'red',         # AT2R            -> korunur
    'B': 'yellow',      # Anjiyotensin II -> korunur
    'C': 'grey',        # BRIL            -> cikarilir
    'H': 'lightblue',   # Fab agir zincir -> cikarilir
    'L': 'lightgreen',  # Fab hafif zincir-> cikarilir
})


**Kavramsal not.** GPCR ailesine ait proteinler kristalizasyona dirençlidir.
Bu güçlüğün aşılması için proteine BRIL veya T4 lizozim gibi bir füzyon bölgesi
eklenmekte ya da yapı bir Fab fragmanı ile kompleks hâlinde kristalize
edilmektedir. Her iki bileşen de deneysel araçtır; fizyolojik ortamda
bulunmamaktadır.

Bu yapıda BRIL ayrı bir zincir olarak deposit edildiğinden çıkarılması doğrudan
zincir silme işlemiyle yapılabilmektedir. Bazı GPCR yapılarında BRIL üçüncü
hücre içi ilmiğin (ICL3) **içine** yerleştirilmiş olduğundan dizinin ortasından
kesilerek çıkarılması gerekir; bunun yapının kaynak makalesine başvurulmadan
anlaşılması mümkün değildir.


---
## 5. Membrana göre yönlendirme

Bu, komut satırı iş akışındaki **en kritik ve en sık atlanan** adımdır.

**Sorun.** PDB dosyasındaki koordinatlar kristal biriminin çerçevesinde
verilmektedir. Bu çerçevenin membran düzlemiyle hiçbir ilişkisi yoktur. `insane`
aracı `-center` ve `-dm` seçenekleriyle proteini kutuya **ortalamakta**, ancak
**döndürmemektedir**. Yönlendirme yapılmazsa protein membrana yatık biçimde
gömülür; transmembran heliksleri lipit çift tabakasına dik olmaz.

6JOD için ölçüm: kristal çerçevesinde proteinin uzun ekseni z ekseniyle
yaklaşık **80 derece** açı yapmaktadır; yani neredeyse membran düzlemine
paraleldir. Doğru yönelimde bu açının küçük olması beklenir.

**Çözüm.** Yapının [OPM](https://opm.phar.umich.edu/) (Orientations of Proteins
in Membranes) veritabanındaki hâli kullanılmaktadır. OPM, PPM algoritmasıyla
proteini membran normaline göre hizalamakta ve membran merkezini `z = 0`
konumuna yerleştirmektedir.

> **Sabah oturumuyla bağlantı.** CHARMM-GUI Membrane Builder da aynı işlemi
> PPM/OPM sunucusunu çağırarak yapmaktadır. Grafik arayüzde arka planda
> gerçekleşen bu adımı burada açıkça yürütmekteyiz.


**Aşağıdaki hücre ne yapıyor?** Yönlendirilmiş yapıyı OPM'den indirmektedir.
Dosyada ayrıca `DUM` adlı sanal atomlar bulunmaktadır; bunlar membranın
hidrofobik sınırlarını işaretlemektedir ve sisteme dâhil edilmeyecektir.


In [ ]:
!wget -q https://opm-assets.storage.googleapis.com/pdb/6jod.pdb -O 6jod_opm.pdb
!ls -lh 6jod_opm.pdb

# OPM dosyasinin basindaki membran kalinligi bilgisi
for l in open('6jod_opm.pdb'):
    if l.startswith('REMARK'):
        print(l.rstrip()); break

# DUM atomlari membran duzlemlerini isaretler
dum_z = sorted({round(float(l[46:54]), 1) for l in open('6jod_opm.pdb')
                if l.startswith('HETATM') and l[17:20].strip() == 'DUM'})
print('Membran duzlemleri (z, Angstrom):', dum_z)
print(f'Hidrofobik kalinlik: {(max(dum_z)-min(dum_z))/10:.2f} nm')


**Aşağıdaki hücre ne yapıyor?** Ham ve yönlendirilmiş yapıların yönelimini
ölçüp karşılaştırmaktadır. `yonelim_olc()` fonksiyonu proteinin en uzun asal
ekseninin z ekseniyle yaptığı açıyı vermektedir.

Beklenen sonuç: RCSB için büyük bir açı (yatık), OPM için küçük bir açı (dik).
GPCR'lerde transmembran demeti hafif eğik olduğundan sıfır beklenmemektedir.


In [ ]:
import numpy as np

def zincir_yaz(kaynak, hedef, zincir='A'):
    """Belirtilen zincirin ATOM kayitlarini yeni bir PDB dosyasina yazar."""
    cryst = ('CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 '
             'P 1           1\n')
    satirlar = [l for l in open(kaynak)
                if l.startswith('ATOM  ') and l[21] == zincir]
    open(hedef, 'w').writelines([cryst] + satirlar + ['END\n'])
    return len(satirlar)

zincir_yaz('6jod_rcsb.pdb', 'karsilastirma_rcsb.pdb')
zincir_yaz('6jod_opm.pdb',  'karsilastirma_opm.pdb')

aci_rcsb = yonelim_olc('karsilastirma_rcsb.pdb', 'RCSB (kristal cercevesi)')
print()
aci_opm  = yonelim_olc('karsilastirma_opm.pdb',  'OPM (membrana yonlendirilmis)')
print()
print(f'Yonelim farki: {abs(aci_rcsb - aci_opm):.1f} derece')
print()
if aci_opm < 30:
    print('DOGRULAMA BASARILI: yonlendirilmis yapi z eksenine yakin hizali.')
else:
    print('UYARI: beklenmeyen yonelim. OPM dosyasi kontrol edilmelidir.')


**Aşağıdaki hücre ne yapıyor?** İki yönelimi yan yana çizerek farkı görsel
olarak göstermektedir. Sol paneldeki yapı membran düzlemine yatık, sağdaki ise
diktir. Turuncu kesikli çizgiler OPM'nin belirlediği membran sınırlarıdır.


In [ ]:
import matplotlib.pyplot as plt

fig, eksenler = plt.subplots(1, 2, figsize=(12, 6))
for eks, (dosya, baslik, aci) in zip(eksenler, [
        ('karsilastirma_rcsb.pdb', 'RCSB - kristal cercevesi', aci_rcsb),
        ('karsilastirma_opm.pdb',  'OPM - membrana yonlendirilmis', aci_opm)]):
    _, _, xyz = koordinat_oku(dosya)
    xyz = xyz - xyz.mean(0)
    eks.scatter(xyz[:, 0], xyz[:, 2], s=3, c='#2E5FA3', alpha=.5, linewidths=0)
    eks.set_title(f'{baslik}\nz ile aci: {aci:.1f} derece')
    eks.set_xlabel('x (nm)'); eks.set_aspect('equal'); eks.grid(alpha=.2)
    eks.set_xlim(-5, 5); eks.set_ylim(-5, 5)
eksenler[0].set_ylabel('z (nm)')
for z in (-1.61, 1.61):
    eksenler[1].axhline(z, color='#E08A2E', ls='--', lw=1.5)
eksenler[1].text(-4.7, 1.75, 'membran siniri (OPM)', color='#E08A2E', fontsize=9)
fig.suptitle('Membrana gore yonlendirmenin etkisi', fontsize=13)
plt.tight_layout()
png = os.path.join(D_GORSEL, '01_yonelim_karsilastirmasi.png')
plt.savefig(png, dpi=150); plt.show()
print('Kaydedildi:', png)


---
## 6. A zincirinin ayıklanması

**Aşağıdaki hücre ne yapıyor?** Yönlendirilmiş OPM dosyasından yalnızca A
zincirini (AT2R) alıp `at2r.pdb` olarak kaydetmektedir. `DUM` işaretçileri,
B zinciri (ligant), su ve hetero gruplar dışarıda bırakılmaktadır.

Bu oturumda sade bir protein–membran sistemi kurulmaktadır; ligant dâhil
edilmemektedir.

Dosyanın başına bir `CRYST1` kaydı eklenmektedir: bazı araçlar bu kaydın
varlığını beklemektedir.


In [ ]:
n = zincir_yaz('6jod_opm.pdb', 'at2r.pdb', 'A')

resids = sorted({int(l[22:26]) for l in open('at2r.pdb')
                 if l.startswith('ATOM  ')})
print(f'A zinciri: {len(resids)} rezidu ({resids[0]}-{resids[-1]}), {n} atom')
print()
yonelim_olc('at2r.pdb', 'Hazirlanan yapi')
print()
kaydet(['6jod_rcsb.pdb', '6jod_opm.pdb', 'at2r.pdb'], D_GIRDI)


**Beklenen sonuç.** 35–340 aralığında 306 rezidü, 2470 atom.
Dizide 312 rezidü bulunmakta olup 341–346 aralığındaki C-terminal uzantı
çözülmemiştir. Zincir içi kopukluk bulunmadığından ilmik modellemesine gerek
duyulmamaktadır.


**Aşağıdaki hücre ne yapıyor?** Ayıklanan yapıyı etkileşimli olarak
göstermektedir. Renkler N-ucundan (mavi) C-ucuna (kırmızı) doğru değişmektedir;
yedi transmembran heliksi ayırt edilebilmektedir.


In [ ]:
yapi_goster('at2r.pdb', stil='karton')


**Aşağıdaki hücre ne yapıyor?** Yapının statik bir kesit çizimini üretip
`gorseller/` klasörüne kaydetmektedir.


In [ ]:
kesit_ciz('at2r.pdb', os.path.join(D_GORSEL, '02_at2r_atomistik.png'),
          'Adim 2 - AT2R atomistik yapi (membrana yonlendirilmis)')


---
## 7. İkincil yapının belirlenmesi

Martini 3'te etkileşim merkezlerinin tipleri ikincil yapıya bağlı olduğundan
`martinize2` her rezidü için bir ikincil yapı ataması gerektirmektedir.

**`-dssp` seçeneği neden kullanılmıyor?** Ubuntu depolarındaki güncel `mkdssp`
sürümü (4.x) girdi dosyasında `CRYST1` kaydı aramakta, `vermouth` tarafından
üretilen geçici dosyada ise bu kayıt bulunmamaktadır. Bu nedenle `-dssp`
kullanımı şu hatayı vermektedir:

```
DSSPError: Expected record CRYST1 but found ATOM
```

**Kullanılan yöntem.** İkincil yapı, kristal yapının kendi `HELIX` ve `SHEET`
kayıtlarından çıkarılıp `-ss` seçeneği ile verilmektedir. Bu atama, yapıyı çözen
araştırmacılar tarafından yapılmış olduğundan güvenilirdir.

> **Dikkat.** OPM dosyasında `HELIX`/`SHEET` kayıtları bulunmamaktadır; bu
> nedenle ikincil yapı **RCSB dosyasından** okunmaktadır. İki dosyada rezidü
> numaralandırması aynıdır.

> Deneysel yapı yerine bir model (örneğin AlphaFold çıktısı) kullanılıyorsa bu
> kayıtlar bulunmayacaktır; o durumda uyumlu bir DSSP sürümü kurulmalı veya
> ikincil yapı `mdtraj` gibi bir kütüphaneyle hesaplanmalıdır.

**Aşağıdaki hücre ne yapıyor?** `HELIX` ve `SHEET` kayıtlarını okuyup her
rezidü için tek harflik bir ikincil yapı kodu (`H` heliks, `E` β-tabaka,
`C` diğer) üretmektedir. Sonuç `SS` değişkenine atanmakta ve 8. bölümde
`martinize2`'ye verilmektedir.


In [ ]:
def ss_cikar(pdb, zincir='A'):
    """PDB dosyasinin HELIX/SHEET kayitlarindan ikincil yapi dizesi uretir."""
    resids = sorted({int(l[22:26]) for l in open(pdb)
                     if l.startswith('ATOM  ') and l[21] == zincir})
    ss = {r: 'C' for r in resids}
    nh = ne = 0
    for l in open(pdb):
        if l.startswith('HELIX') and l[19] == zincir:
            nh += 1
            for r in range(int(l[21:25]), int(l[33:37]) + 1):
                if r in ss: ss[r] = 'H'
        elif l.startswith('SHEET') and l[21] == zincir:
            ne += 1
            for r in range(int(l[22:26]), int(l[33:37]) + 1):
                if r in ss: ss[r] = 'E'
    dize = ''.join(ss[r] for r in resids)
    print(f'HELIX kaydi: {nh} | SHEET kaydi: {ne}')
    print(f'Dize uzunlugu: {len(dize)} (rezidu sayisi: {len(resids)})')
    print(f"Dagilim -> H: {dize.count('H')} "
          f"(%{100*dize.count('H')/len(dize):.0f})  "
          f"E: {dize.count('E')}  C: {dize.count('C')}")
    return dize

SS = ss_cikar('6jod_rcsb.pdb', 'A')
print()
for i in range(0, len(SS), 60):
    print(f'{i+1:>4}  {SS[i:i+60]}')


**Beklenen sonuç.** 306 karakterlik bir dize; heliks oranı yaklaşık %84.
Yedi transmembran heliksi ve hücre dışı ikinci ilmikteki (ECL2) kısa β-firkete
(`E` bloğu) bu dizede görülebilmektedir. Bu dağılım bir GPCR için beklenen
yapıya uymaktadır.


---
## 8. `martinize2` — elastik ağ ile kaba-taneli model

**Aşağıdaki hücre ne yapıyor?** Atomistik proteini Martini 3 etkileşim
merkezlerine dönüştürmekte ve yapıyı korumak için bir elastik ağ
eklemektedir.

| Parametre | İşlevi |
|---|---|
| `-ff martini3001` | Martini 3 kuvvet alanı |
| `-ss` | İkincil yapı ataması (7. bölümde üretildi) |
| `-elastic` | Elastik ağ tanımlanması |
| `-ef 700` | Yay kuvvet sabiti (kJ mol⁻¹ nm⁻²) |
| `-el 0.5 -eu 0.9` | Yay tanımlanacak mesafe aralığı (nm) |
| `-cys auto` | Disülfit köprülerinin otomatik belirlenmesi |

Çıktı olarak `at2r_cg.pdb` (koordinatlar), `topol.top` ve `molecule_0.itp`
(topoloji) üretilmektedir.


In [ ]:
!martinize2 \
  -f at2r.pdb \
  -o topol.top \
  -x at2r_cg.pdb \
  -ff martini3001 \
  -ss {SS} \
  -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 \
  -cys auto \
  -maxwarn 10


Çıktıdaki `Disulfide bridge found between residues A-CYS1 and A-CYS256` ile
`A-CYS83 and A-CYS161` iletileri `martinize2`'nin kendi iç numaralandırmasını
kullanmaktadır. PDB numaralandırmasına çevrildiğinde bunlar **Cys35–Cys290** ve
**Cys117–Cys195** köprülerine karşılık gelmektedir; ikisi de yapının `SSBOND`
kayıtlarıyla uyumludur.


**Aşağıdaki hücre ne yapıyor?** Atom sayısından etkileşim merkezi sayısına
indirgeme oranını hesaplamakta ve üretilen dosyaları Drive'a kaydetmektedir.


In [ ]:
aa = sum(1 for l in open('at2r.pdb')    if l.startswith('ATOM'))
cg = sum(1 for l in open('at2r_cg.pdb') if l.startswith(('ATOM','HETATM')))
print(f'Atomistik model  : {aa:>6} atom')
print(f'Kaba-taneli model: {cg:>6} etkilesim merkezi')
print(f'Indirgeme orani  : {aa/cg:.1f}')
print()
kaydet(['at2r_cg.pdb', 'molecule_*.itp', 'topol.top'], D_CIKTI)


**Aşağıdaki hücre ne yapıyor?** Kaba-taneli yapıyı etkileşimli göstermektedir.
Mavi küreler omurga (BB), gri küreler yan zincir (SC) merkezleridir. Atomistik
görünümle karşılaştırınız: molekülün genel şekli korunmakta, atomik ayrıntı
kaybolmaktadır.


In [ ]:
yapi_goster('at2r_cg.pdb', stil='cg_protein')


**Aşağıdaki hücre ne yapıyor?** İki çözünürlüğü yan yana çizip PNG olarak
kaydetmektedir.


In [ ]:
res_aa, ad_aa, xyz_aa = koordinat_oku('at2r.pdb')
res_cg, ad_cg, xyz_cg = koordinat_oku('at2r_cg.pdb')

fig, eksenler = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
for eks, (xyz, etiket, renk, boyut) in zip(eksenler, [
        (xyz_aa, f'Atomistik ({len(xyz_aa):,} atom)', '#2E5FA3', 4),
        (xyz_cg, f'Martini 3 ({len(xyz_cg):,} merkez)', '#C0392B', 22)]):
    eks.scatter(xyz[:, 0], xyz[:, 2], s=boyut, c=renk, alpha=.6, linewidths=0)
    eks.set_title(etiket); eks.set_xlabel('x (nm)')
    eks.set_aspect('equal'); eks.grid(alpha=.2)
eksenler[0].set_ylabel('z (nm)')
fig.suptitle('Adim 3 - cozunurluk indirgeme', fontsize=13)
plt.tight_layout()
png = os.path.join(D_GORSEL, '03_atomistik_vs_kaba_taneli.png')
plt.savefig(png, dpi=150); plt.show()
print('Kaydedildi:', png)


---
## 9. Yapısal kısıt modelleri: elastik ağ, Gō-Martini ve OLIVES

Martini kuvvet alanı, etkileşim merkezleri arasındaki potansiyeller aracılığıyla
proteinin üçüncül yapısını **koruyamamaktadır**. Kısıt uygulanmayan bir Martini
proteini simülasyon sırasında açılır. Bu nedenle yapıya dışarıdan bir kısıt
eklenmesi zorunludur.

Bugün kullanılan yöntem elastik ağdır. Ancak alanda yaygın kullanılan üç
yaklaşım bulunmaktadır ve hangisinin seçileceği araştırma sorusuna bağlıdır.

### Karşılaştırma

| | **Elastik ağ** | **Gō-Martini** | **OLIVES** |
|---|---|---|---|
| Kısıtın türü | Harmonik yaylar (`[ bonds ]`) | Lennard-Jones kontakları (sanal bölgeler üzerinden) | Hidrojen bağı temelli LJ kontakları |
| Kontak seçimi | Mesafe ölçütü (0.5–0.9 nm arası tüm çiftler) | Yapısal kontak haritası (örtüşme + rCSU ölçütü) | Atomistik yapıdan çıkarılan hidrojen bağı ağı |
| Kısıt kopabilir mi? | **Hayır** — yaylar kalıcıdır | **Evet** — LJ kontakları kopup yeniden kurulabilir | **Evet** |
| İkincil yapı | Sabit, `-ss` ile dayatılır | Sabit | **Dinamik olabilir** |
| Katlanma / açılma incelenebilir mi? | Hayır | Kısmen (mekanik açılma, alt birim ayrışması) | Evet |
| Çoklu alt birim / kompleks | Zayıf | İyi | İyi |
| Kurulum karmaşıklığı | En düşük | Orta | Orta–yüksek |
| `martinize2` desteği | `-elastic` | `-go` (v0.15'te dâhili) | Ayrı betik, sonradan uygulanır |

### Hangisi ne zaman?

- **Elastik ağ.** Protein yapısının sabit kalması beklenen ve ilgi odağının
  **çevre** olduğu çalışmalar: lipit–protein etkileşimleri, membran içi difüzyon,
  oligomerleşme, kalabalık ortam. Yayımlanmış Martini membran proteini
  çalışmalarının büyük bölümü bu yaklaşımı kullanmaktadır. **Bu kursun konusu
  budur.**
- **Gō-Martini.** Kısıtların kopabilmesinin gerektiği durumlar: mekanik açılma,
  alt birim ayrışması, kısmi yapısal esneklik, allosterik geçişlerin kaba
  incelenmesi.
- **OLIVES.** İkincil yapının kendisinin değişebilmesi gereken çalışmalar;
  ayrıca nükleik asitler ve büyük kompleksler.

### Kaynaklar

- Souza ve ark. (2025). *GōMartini 3.* Nature Communications, 16.
  [doi:10.1038/s41467-025-58719-0](https://doi.org/10.1038/s41467-025-58719-0)
- [OLIVES — Martini Force Field Initiative](https://github.com/Martini-Force-Field-Initiative/OLIVES)
- [Notes and Limitations — cgmartini.nl](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut4.html)


---
## 9. Yapısal kısıt modelleri: elastik ağ, Gō-Martini ve OLIVES

Martini kuvvet alanı, etkileşim merkezleri arasındaki potansiyeller aracılığıyla
proteinin üçüncül yapısını **koruyamamaktadır**. Kısıt uygulanmayan bir Martini
proteini simülasyon sırasında açılır. Bu nedenle yapıya dışarıdan bir kısıt
eklenmesi zorunludur.

Alanda yaygın kullanılan üç yaklaşım bulunmaktadır. Bu bölümde üçü de aynı
protein üzerinde uygulanıp karşılaştırılmaktadır.

| | **Elastik ağ** | **Gō-Martini** | **OLIVES** |
|---|---|---|---|
| Kısıtın türü | Harmonik yaylar (`[ bonds ]`) | Lennard-Jones kontakları (sanal bölgeler üzerinden) | Lennard-Jones kontakları (`[ pairs ]`) |
| Kontak ölçütü | Mesafe (0.5–0.9 nm arası tüm çiftler) | Yapısal kontak haritası (örtüşme + rCSU) | **Hidrojen bağı ağı** |
| Neyi bağlar? | Yalnızca omurga | Yalnızca omurga | **Omurga ve yan zincirler** |
| Kısıt kopabilir mi? | **Hayır** — yaylar kalıcıdır | **Evet** | **Evet** |
| İkincil yapı | Sabit, `-ss` ile dayatılır | Sabit, `-ss` ile dayatılır | **Dinamik** (`-ss` gerekmez) |
| Katlanma / açılma | İncelenemez | Kısmen | İncelenebilir |
| Çoklu alt birim | Zayıf | İyi | İyi |
| Kurulum | En basit | Orta | Orta |
| Uygulama | `martinize2 -elastic` | `martinize2 -go` | Ayrı betik, sonradan |

### Hangisi ne zaman?

- **Elastik ağ.** Protein yapısının sabit kalması beklenen, ilgi odağının
  **çevre** olduğu çalışmalar: lipit–protein etkileşimleri, membran içi difüzyon,
  oligomerleşme, kalabalık ortam. Yayımlanmış Martini membran proteini
  çalışmalarının büyük bölümü bu yaklaşımı kullanmaktadır. **Bu kursun ana
  akışında kullanılan yöntem budur.**
- **Gō-Martini.** Kısıtların kopabilmesinin gerektiği durumlar: mekanik açılma,
  alt birim ayrışması, kısmi yapısal esneklik.
- **OLIVES.** İkincil yapının kendisinin değişebilmesi gereken çalışmalar;
  ayrıca protein kompleksleri ve nükleik asitler.

### Kaynaklar

- Souza ve ark. (2025). *GōMartini 3.* Nature Communications, 16.
  [doi:10.1038/s41467-025-58719-0](https://doi.org/10.1038/s41467-025-58719-0)
- Pedersen ve ark. (2024). *OLIVES.* J. Chem. Theory Comput.
  [doi:10.1021/acs.jctc.4c00553](https://doi.org/10.1021/acs.jctc.4c00553)
- [OLIVES deposu](https://github.com/Martini-Force-Field-Initiative/OLIVES)


**Aşağıdaki hücre ne yapıyor?** OLIVES için gereken paketleri kurup betiği
indirmektedir. `mdtraj`, `networkx` ve `pandas` yalnızca bu bölüm için
gerekmektedir; kurulum yaklaşık bir dakika sürer.


In [ ]:
%%capture
!pip install -q mdtraj networkx pandas
!wget -q https://raw.githubusercontent.com/Martini-Force-Field-Initiative/OLIVES/main/OLIVES_v2.1.1b_M3.0.0.py -O OLIVES.py


**Aşağıdaki hücre ne yapıyor?** Aynı proteini Gō-Martini modeliyle
kaba-taneleştirmektedir. Yalnızca bir bayrak değişmektedir: `-elastic` yerine
`-go`. `martinize2` v0.15 kontak haritasını **kendisi hesaplamaktadır**; harici
bir sunucuya gerek yoktur. İşlem birkaç dakika sürebilir.


In [ ]:
!mkdir -p model_go && cp at2r.pdb model_go/
%cd model_go
!martinize2 -f at2r.pdb -o go.top -x at2r_go.pdb -ff martini3001 \
  -ss {SS} -go -go-write-file go_contacts.out -cys auto -maxwarn 20 2>&1 | tail -4
%cd ..


**Aşağıdaki hücre ne yapıyor?** OLIVES modelini uygulamaktadır. İki adımdan
oluşur:

1. `martinize2` **`-ss` bayrağı olmadan** çalıştırılır — OLIVES'ın temel iddiası
   ikincil yapının dayatılmasına gerek bırakmamasıdır. `-scfix` bayrağı yan
   zincir konformasyonlarını sabitler.
2. `OLIVES.py` betiği, kaba-taneli yapıdaki hidrojen bağı ağını belirleyip
   karşılık gelen kontakları doğrudan topolojiye ekler.

Çıktıdaki `Secondary`/`Tertiary HB pairs` satırları, ikincil yapı içi ve
üçüncül yapı arası kontak sayılarını vermektedir.


In [ ]:
!mkdir -p model_olives && cp at2r.pdb OLIVES.py model_olives/
%cd model_olives
!martinize2 -f at2r.pdb -x at2r_olives.pdb -o olives.top \
  -ff martini3001 -scfix -cys auto -maxwarn 20 2>&1 | tail -2
!python3 OLIVES.py -c at2r_olives.pdb -i molecule_0.itp 2>&1 | grep -E 'pairs|Detected|Done'
%cd ..


**Aşağıdaki hücre ne yapıyor?** Üç modelin topolojilerinden kısıt çiftlerini
ayrıştıran fonksiyonları tanımlamaktadır. Her model kısıtlarını farklı bir yere
yazdığından ayrı ayrı okunmaları gerekmektedir:

| Model | Kısıtların yeri |
|---|---|
| Elastik ağ | `molecule_0.itp` → `[ bonds ]` → `; Rubber band` bloğu |
| Gō-Martini | `go_nbparams.itp` → sanal bölge adları rezidü numarasına çevrilir |
| OLIVES | `molecule_0.itp` → OLIVES tarafından eklenen `[ pairs ]` blokları |


In [ ]:
def cg_oku(pdb):
    """Martini CG PDB -> (koordinat nm, atom adi, rezidu no)"""
    xyz, ad, resid = [], [], []
    for l in open(pdb):
        if l.startswith(('ATOM  ', 'HETATM')):
            xyz.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
            ad.append(l[12:16].strip()); resid.append(int(l[22:26]))
    return np.array(xyz) / 10.0, np.array(ad), np.array(resid)


def en_ciftleri(itp):
    """Elastik ag yaylari: '; Rubber band' yorumundan sonraki bloklar."""
    sat = open(itp).read().splitlines()
    try:
        i = next(k for k, l in enumerate(sat) if l.strip() == '; Rubber band')
    except StopIteration:
        return []
    ciftler = []
    for l in sat[i+1:]:
        s = l.strip()
        if s.startswith('[') or s.startswith(';'):
            break
        if s:
            p = s.split()
            if len(p) >= 2:
                ciftler.append((int(p[0]) - 1, int(p[1]) - 1))
    return ciftler


def go_ciftleri(nbparams, itp, cg_pdb):
    """Sanal bolge adlarini (molecule_N) rezidu no'ya, oradan BB indeksine cevirir."""
    res_of = {}; icinde = False
    for l in open(itp):
        s = l.strip()
        if s.startswith('['):
            icinde = (s.strip('[] ') == 'atoms'); continue
        if icinde and s and not s.startswith(';'):
            p = s.split()
            if len(p) >= 5 and p[4] == 'CA':
                res_of[p[1]] = int(p[2])
    xyz, ad, resid = cg_oku(cg_pdb)
    bb = {}
    for k, (a, r) in enumerate(zip(ad, resid)):
        if a == 'BB' and r not in bb:
            bb[r] = k
    ciftler = []
    for l in open(nbparams):
        s = l.strip()
        if not s or s.startswith((';', '[')):
            continue
        p = s.split()
        r1, r2 = res_of.get(p[0]), res_of.get(p[1])
        if r1 in bb and r2 in bb:
            ciftler.append((bb[r1], bb[r2]))
    return ciftler


def olives_ciftleri(itp):
    """OLIVES tarafindan eklenen [ pairs ] bloklari."""
    sat = open(itp).read().splitlines()
    basla = None
    for k, l in enumerate(sat):
        if 'OLIVES' in l and 'pairs' in l.lower():
            basla = k; break
    if basla is None:
        return []
    ciftler = []; icinde = False
    for l in sat[basla:]:
        s = l.strip()
        if s.startswith('['):
            icinde = (s.strip('[] ') == 'pairs'); continue
        if s and not s.startswith(';') and icinde:
            p = s.split()
            if len(p) >= 2 and p[0].isdigit():
                ciftler.append((int(p[0]) - 1, int(p[1]) - 1))
    return ciftler

print('Ayristirma fonksiyonlari hazir.')


**Aşağıdaki hücre ne yapıyor?** Üç modelin kısıt ağını dört panel hâlinde
çizmektedir. Gri çizgi omurga izini, renkli çizgiler kısıtları göstermektedir.

**Dikkat edilecek noktalar:**

- Elastik ağ en yoğun ağı üretmektedir; protein âdeta bir kafes içine alınmıştır
- Gō-Martini yaklaşık yarı sayıda, daha seçici kontak kurmaktadır
- OLIVES en az sayıda kontak kullanmakta ve **yan zincirleri de bağlamaktadır**;
  omurgadan dışarı uzanan kırmızı çizgiler bunlardır

Bu karşılaştırma, Pedersen ve ark. (2024) makalesinin 3. şeklindeki gösterimin
6JOD için üretilmiş hâlidir.


In [ ]:
from matplotlib.collections import LineCollection

def ag_paneli(eks, xyz, ad, resid, ciftler, renk, baslik, i=0, j=2):
    xyz = xyz - xyz.mean(0)
    bb = sorted([k for k, a in enumerate(ad) if a == 'BB'], key=lambda k: resid[k])
    eks.plot(xyz[bb, i], xyz[bb, j], color='#909090', lw=2.0,
             zorder=1, solid_capstyle='round')
    if ciftler:
        segler = [[(xyz[a, i], xyz[a, j]), (xyz[b, i], xyz[b, j])]
                  for a, b in ciftler]
        eks.add_collection(LineCollection(segler, colors=renk,
                                          linewidths=0.45, alpha=0.65, zorder=2))
    eks.set_title(baslik, fontsize=11, fontweight='bold')
    eks.set_aspect('equal'); eks.axis('off')


xyz_en, ad_en, r_en = cg_oku('at2r_cg.pdb')
xyz_go, ad_go, r_go = cg_oku('model_go/at2r_go.pdb')
xyz_ol, ad_ol, r_ol = cg_oku('model_olives/at2r_olives.pdb')

en = en_ciftleri(sorted(glob.glob('molecule_*.itp'))[0])
go = go_ciftleri('model_go/go_nbparams.itp', 'model_go/molecule.itp',
                 'model_go/at2r_go.pdb')
ol = olives_ciftleri('model_olives/molecule_0.itp')

fig, eks = plt.subplots(1, 4, figsize=(17, 5.2))

ca = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
               for l in open('at2r.pdb')
               if l.startswith('ATOM  ') and l[12:16].strip() == 'CA']) / 10.0
ca = ca - ca.mean(0)
eks[0].plot(ca[:, 0], ca[:, 2], color='#909090', lw=2.0, solid_capstyle='round')
eks[0].set_title('Atomistik\n(kisit yok)', fontsize=11, fontweight='bold')
eks[0].set_aspect('equal'); eks[0].axis('off')

ag_paneli(eks[1], xyz_en, ad_en, r_en, en, '#2E5FA3', f'Elastik ag\n({len(en)} yay)')
ag_paneli(eks[2], xyz_go, ad_go, r_go, go, '#E08A2E', f'GoMartini\n({len(go)} kontak)')
ag_paneli(eks[3], xyz_ol, ad_ol, r_ol, ol, '#C0392B', f'OLIVES\n({len(ol)} kontak)')
for e in eks:
    e.set_xlim(-3.6, 3.6); e.set_ylim(-4.6, 4.6)

fig.suptitle('6JOD A zinciri (AT2R) - yapisal kisit modellerinin karsilastirilmasi',
             fontsize=13)
plt.tight_layout()
png = os.path.join(D_GORSEL, '04_kisit_aglari.png')
plt.savefig(png, dpi=150, facecolor='white'); plt.show()
print('Kaydedildi:', png)
print()
print(f'Elastik ag : {len(en):>5} yay')
print(f'GoMartini  : {len(go):>5} kontak')
print(f'OLIVES     : {len(ol):>5} kontak')


**Aşağıdaki hücre ne yapıyor?** Üç modelin topolojilerini sayısal olarak
karşılaştırmaktadır. Kısıtların topolojide **nereye** yazıldığına dikkat ediniz:
elastik ağ `[ bonds ]` bölümüne yay ekler; Gō modeli her rezidü için bir sanal
bölge tanımlayıp kontakları ayrı bir dosyada nonbonded parametre olarak verir;
OLIVES ise `[ pairs ]` bölümünü kullanır.


In [ ]:
def bolum_say(itp, bolum):
    if not os.path.exists(itp):
        return 0
    n = 0; icinde = False
    for l in open(itp):
        s = l.strip()
        if s.startswith('['):
            icinde = (s.strip('[] ') == bolum); continue
        if icinde and s and not s.startswith(';'):
            n += 1
    return n

def dosya_say(p):
    if not os.path.exists(p):
        return 0
    return sum(1 for l in open(p)
               if l.strip() and not l.strip().startswith((';', '[')))

en_itp = sorted(glob.glob('molecule_*.itp'))[0]
go_itp = 'model_go/molecule.itp'
ol_itp = 'model_olives/molecule_0.itp'

print(f"{'':<28}{'Elastik ag':>12}{'GoMartini':>12}{'OLIVES':>12}")
print('-' * 64)
print(f"{'CG parcacik':<28}"
      f"{len(xyz_en):>12}{len(xyz_go):>12}{len(xyz_ol):>12}")
for b in ('bonds', 'constraints', 'angles', 'dihedrals',
          'exclusions', 'virtual_sitesn', 'pairs'):
    print(f'{b:<28}{bolum_say(en_itp, b):>12}'
          f'{bolum_say(go_itp, b):>12}{bolum_say(ol_itp, b):>12}')
print(f"{'go_nbparams (kontak)':<28}{0:>12}"
      f"{dosya_say('model_go/go_nbparams.itp'):>12}{0:>12}")
print('-' * 64)
print(f"{'KISIT SAYISI':<28}{len(en):>12}{len(go):>12}{len(ol):>12}")
print()
kaydet(['model_go/*.itp', 'model_go/go.top', 'model_go/at2r_go.pdb',
        'model_olives/molecule_0.itp', 'model_olives/at2r_olives.pdb'], D_CIKTI)


> **Not.** Oturumun devamında **elastik ağ** modeliyle ilerlenecektir. Gō ve
> OLIVES modellerinin tam bir membran sistemine yerleştirilmesi ek topoloji
> düzenlemeleri gerektirmektedir (Gō için `go_atomtypes.itp` ve
> `go_nbparams.itp` dosyalarının kuvvet alanının `[ atomtypes ]` ve
> `[ nonbond_params ]` bölümlerine doğru sırayla eklenmesi). Bu işlemler kurs
> süresinin dışında bırakılmış olup üretilen dosyalar Drive'a kaydedilmektedir.


---
## 10. Martini 3 kuvvet alanı dosyaları

**Aşağıdaki hücre ne yapıyor?** Martini 3'ün genel parametre dosyalarını
indirmektedir. `martinize2` yalnızca proteinin topolojisini üretti; etkileşim
merkezi tanımları, lipitler, çözücü ve iyon parametreleri ayrıca gereklidir.

Ana parametre dosyasının boyutu yaklaşık 16 MB olup indirme bir dakika
sürebilir. Bu dosyalar 18. bölümde simülasyon paketine dâhil edilecektir.

Kaynak: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)


In [ ]:
BASE = 'https://raw.githubusercontent.com/marrink-lab/martini-forcefields/main/martini_forcefields/regular/v3.0.0/gmx_files'
FF_DOSYALARI = ['martini_v3.0.0.itp',
                'martini_v3.0.0_solvents_v1.itp',
                'martini_v3.0.0_ions_v1.itp',
                'martini_v3.0.0_phospholipids_v1.itp']
for f in FF_DOSYALARI:
    !wget -q {BASE}/{f} -O {f}
!ls -lh martini_v3.0.0*.itp


**Aşağıdaki hücre ne yapıyor?** Kuvvet alanındaki iyon molekül tiplerini
listelemektedir. 12. bölümde bu adların neden önemli olduğu görülecektir.


In [ ]:
!grep -A2 'moleculetype' martini_v3.0.0_ions_v1.itp | grep -E '^(NA|CL|K|CA) ' | head


---
## 11. `insane` ile membran, çözücü ve iyon eklenmesi

**Aşağıdaki hücre ne yapıyor?** Kaba-taneli proteinin çevresine lipit çift
tabakası inşa etmekte, kutuyu çözücü ve iyonla doldurmaktadır.

| Parametre | İşlevi |
|---|---|
| `-box 12,12,14` | Kutu boyutları (nm) |
| `-l POPC:1` | Lipit bileşimi |
| `-sol W` | Martini standart su modeli (bir merkez ≈ dört su molekülü) |
| `-salt 0.15` | 0.15 M NaCl |
| `-center` | Proteinin kutuya ortalanması |
| `-dm 0` | Proteinin membran düzlemine göre z ötelemesi |

Çok bileşenli membran için: `-l POPC:7 -l POPE:2 -l CHOL:1`

> **Hatırlatma.** `insane` proteini ortalamakta ancak **döndürmemektedir**.
> Doğru yönelim 5. bölümde OPM ile sağlanmıştır.


In [ ]:
!insane \
  -f at2r_cg.pdb \
  -o sistem_ham.gro \
  -p sistem_insane.top \
  -pbc square \
  -box 12,12,14 \
  -l POPC:1 \
  -sol W \
  -salt 0.15 \
  -center \
  -dm 0


**Aşağıdaki hücre ne yapıyor?** `insane`'in ürettiği topolojiyi ve toplam
parçacık sayısını göstermektedir. Topolojideki iyon adlarına (`NA+`, `CL-`)
dikkat ediniz — bir sonraki bölümün konusu budur.


In [ ]:
print('--- insane tarafindan uretilen topoloji ---')
print(open('sistem_insane.top').read())
n = int(open('sistem_ham.gro').read().splitlines()[1])
print(f'Toplam parcacik sayisi: {n:,}')


---
## 12. İyon adlarının düzeltilmesi

**Bu adım atlanırsa `gmx grompp` şu hatayı verir:**

```
ERROR 1 [file sistem.top]:
  No such moleculetype NA+
```

**Nedeni.** `insane` aracı Martini 2 döneminde geliştirilmiş olup iyonları `NA+`
ve `CL-` adlarıyla üretmektedir. Martini 3 kuvvet alanı ise aynı iyonları `NA`
ve `CL` adlarıyla tanımlamaktadır (10. bölümdeki çıktıda görülebilir). Adlar
eşleşmediğinden GROMACS molekül tipini bulamamaktadır.

**Aşağıdaki hücre ne yapıyor?** Koordinat dosyasındaki iyon rezidü ve atom
adlarını dönüştürmektedir. GRO biçiminin sabit sütun genişlikleri korunmaktadır.

> Bu, `insane` ile Martini 3'ün birlikte kullanımında karşılaşılan en yaygın
> sorundur. Kendi sistemlerinizi kurarken de aynı düzeltmeyi yapmanız
> gerekecektir.


In [ ]:
ESLEME = {'NA+': 'NA', 'CL-': 'CL', 'K+': 'K'}

def iyon_adlarini_duzelt(gro_girdi, gro_cikti):
    """GRO dosyasindaki Martini 2 iyon adlarini Martini 3 adlarina cevirir."""
    satirlar = open(gro_girdi).read().splitlines()
    n = int(satirlar[1])
    yeni = satirlar[:2]
    degisen = 0
    for s in satirlar[2:2+n]:
        rn, an = s[5:10].strip(), s[10:15].strip()
        if rn in ESLEME or an in ESLEME:
            s = s[:5] + f'{ESLEME.get(rn, rn):<5}' + f'{ESLEME.get(an, an):>5}' + s[15:]
            degisen += 1
        yeni.append(s)
    yeni += satirlar[2+n:]          # kutu vektoru satiri
    open(gro_cikti, 'w').write('\n'.join(yeni) + '\n')
    return degisen

n_degisen = iyon_adlarini_duzelt('sistem_ham.gro', 'sistem.gro')
print(f'{n_degisen} iyon parcaciginin adi duzeltildi (NA+ -> NA, CL- -> CL)')

kalan = sum(1 for s in open('sistem.gro').read().splitlines()[2:-1]
            if s[5:10].strip() in ESLEME)
print('Duzeltilmemis kalan iyon:', kalan)
assert kalan == 0, 'Iyon adlari tam donusturulemedi'


---
## 13. Topolojinin tamamlanması

**Aşağıdaki hücre ne yapıyor?** Eksiksiz bir `sistem.top` dosyası yazmaktadır.

`insane`'in ürettiği topolojide üç eksiklik bulunmaktadır:

1. Kuvvet alanı `#include` yönergeleri yanlış (`martini.itp` diye tek bir dosya
   aranmakta)
2. Protein molekülünün adı `martinize2` çıktısıyla eşleşmemekte
3. İyon adları düzeltilmemiş

Yönerge sırası önemlidir: önce genel kuvvet alanı tanımları, ardından molekül
topolojileri yer almalıdır.


In [ ]:
prot_itp = sorted(glob.glob('molecule_*.itp'))[0]

ham = open('sistem_insane.top').read()
mols = [m.strip() for m in ham.split('[ molecules ]')[1].strip().splitlines()
        if m.strip() and not m.strip().startswith(';')]

# Protein molekul adinin martinize2 ciktisiyla eslestirilmesi
prot_ad = None
satirlar = open(prot_itp).read().splitlines()
for i, l in enumerate(satirlar):
    if l.strip().startswith('[ moleculetype ]'):
        for l2 in satirlar[i+1:]:
            if l2.strip() and not l2.strip().startswith(';'):
                prot_ad = l2.split()[0]; break
        break
print('Protein molekul adi:', prot_ad)

duzeltilmis = []
for m in mols:
    ad, sayi = m.split()[0], m.split()[1]
    if ad.lower().startswith('protein'):
        ad = prot_ad
    ad = ESLEME.get(ad, ad)
    duzeltilmis.append(f'{ad:<12} {sayi}')

top = f'''; Biyofizik 2026 Kursu - Oturum 4
; AT2R (6JOD A zinciri, OPM ile yonlendirilmis) - Martini 3 - POPC membran

#include "martini_v3.0.0.itp"
#include "martini_v3.0.0_solvents_v1.itp"
#include "martini_v3.0.0_ions_v1.itp"
#include "martini_v3.0.0_phospholipids_v1.itp"
#include "{prot_itp}"

[ system ]
AT2R in POPC membrane (Martini 3)

[ molecules ]
''' + '\n'.join(duzeltilmis) + '\n'

open('sistem.top', 'w').write(top)
print()
print(top)
kaydet(['sistem.gro', 'sistem.top', 'sistem_insane.top'], D_CIKTI)


---
## 14. Simülasyon parametre dosyaları (`.mdp`)

Bir simülasyon üç aşamada yürütülür ve her aşamanın kendi parametre dosyası
vardır:

| Dosya | Aşama | Amaç |
|---|---|---|
| `em.mdp` | Enerji minimizasyonu | Kurulumdaki çakışmaların giderilmesi |
| `eq.mdp` | Dengeleme | Sıcaklık ve basıncın hedef değerlere getirilmesi |
| `md.mdp` | Üretim | Analiz edilecek trajektorinin üretilmesi |

**Aşağıdaki hücre ne yapıyor?** Bu üç dosyayı Martini 3 için önerilen standart
ayarlarla yazmaktadır. Öne çıkan parametreler:

- `dt = 0.020` (üretim) — Martini'de 20 fs kullanılabilmektedir; atomistik
  simülasyonlarda bu değer 2 fs'tir
- `epsilon_r = 15` — Martini'nin örtük kutuplanma yaklaşımı
- `coulombtype = reaction-field`, `rcoulomb = rvdw = 1.1`
- `Pcoupltype = semiisotropic` — membran sistemleri için zorunludur; membran
  düzlemi ile normali ayrı ayrı ölçeklenmelidir
- `tc-grps = Protein Lipid Solvent` — bu gruplar 15. bölümde tanımlanacaktır

> Ayarlar cgmartini.nl tarafından dağıtılan güncel Martini `.mdp` kümesine
> dayanmaktadır.


In [ ]:
ORTAK = '''
; Neighbourlist settings
cutoff-scheme            = Verlet
ns_type                  = grid
pbc                      = xyz
verlet-buffer-tolerance  = -1
rlist                    = 1.35

; Non-bonded interaction (Martini 3)
coulombtype              = reaction-field
rcoulomb                 = 1.1
epsilon_r                = 15
epsilon_rf               = 0
vdw_type                 = cutoff
vdw-modifier             = Potential-shift-verlet
rvdw                     = 1.1

; Constraints
constraints              = none
constraint_algorithm     = Lincs
'''

EM = '''; Biyofizik 2026 Kursu - Martini 3 enerji minimizasyonu
title                    = Martini energy minimization
define                   = -DFLEXIBLE

integrator               = steep
dt                       = 0.01
nsteps                   = 1000
nstcomm                  = 100
nstlist                  = 20

gen_vel                  = no
''' + ORTAK

EQ = '''; Biyofizik 2026 Kursu - Martini 3 dengeleme (100 ps)
title                    = Martini equilibration

integrator               = md
dt                       = 0.010
nsteps                   = 10000        ; 0.01 ps x 10.000 = 100 ps
nstcomm                  = 100
nstlist                  = 10

nstxout                  = 0
nstvout                  = 0
nstfout                  = 0
nstlog                   = 1000
nstenergy                = 1000
nstxout-compressed       = 1000
compressed-x-precision   = 100

; Termostat
tcoupl                   = v-rescale
tc-grps                  = Protein Lipid Solvent
tau_t                    = 1.0 1.0 1.0
ref_t                    = 310 310 310

; Barostat - membran sistemlerinde semiisotropic zorunludur
Pcoupl                   = c-rescale
Pcoupltype               = semiisotropic
tau_p                    = 4.0
compressibility          = 3e-4 3e-4
ref_p                    = 1.0 1.0

gen_vel                  = yes
gen_temp                 = 310
gen_seed                 = -1
''' + ORTAK

MD = '''; Biyofizik 2026 Kursu - Martini 3 uretim simulasyonu (1 us)
title                    = Martini production

integrator               = md
dt                       = 0.020        ; 20 fs - Martini icin tipik
nsteps                   = 50000000     ; 0.02 ps x 50.000.000 = 1 us
nstcomm                  = 100
nstlist                  = 20

nstxout                  = 0
nstvout                  = 0
nstfout                  = 0
nstlog                   = 50000
nstenergy                = 50000
nstxout-compressed       = 50000
compressed-x-precision   = 100

; Termostat
tcoupl                   = v-rescale
tc-grps                  = Protein Lipid Solvent
tau_t                    = 1.0 1.0 1.0
ref_t                    = 310 310 310

; Barostat
Pcoupl                   = c-rescale
Pcoupltype               = semiisotropic
tau_p                    = 4.0
compressibility          = 3e-4 3e-4
ref_p                    = 1.0 1.0

gen_vel                  = no
''' + ORTAK

for ad, icerik in [('em.mdp', EM), ('eq.mdp', EQ), ('md.mdp', MD)]:
    open(ad, 'w').write(icerik)
    print(f'{ad:<10} {len(icerik.splitlines()):>3} satir')

kaydet(['em.mdp', 'eq.mdp', 'md.mdp'], D_CIKTI)


**Aşağıdaki hücre ne yapıyor?** Termostat gruplarını tanımlayan `index.ndx`
dosyasını doğrudan koordinat dosyasından üretmektedir.

`eq.mdp` ve `md.mdp` dosyaları `tc-grps = Protein Lipid Solvent` satırını
içermektedir; bu adlarda üç grubun tanımlı olması gerekir. GROMACS'in öntanımlı
grupları arasında `Lipid` ve `Solvent` bulunmadığından bunlar oluşturulmalıdır.

**Neden `gmx make_ndx` kullanılmıyor?** `make_ndx` etkileşimli bir araçtır ve
komutları grup **numaralarına** göre yorumlar. `name 0 Protein` gibi bir komut,
yeni oluşturulan grubu değil, 0 numaralı grubu (yani `System`'i) yeniden
adlandırır. Grup numaraları sisteme göre değiştiğinden bu yaklaşım sessizce
yanlış gruplar üretebilmektedir. Aşağıdaki fonksiyon grupları rezidü adlarından
doğrudan belirlediği için bu belirsizliği ortadan kaldırmakta ve sonucu
doğrulamaktadır.

**Neden ayrı gruplar?** Protein, lipit ve çözücünün ısı kapasiteleri farklı
olduğundan termostatın her birine ayrı uygulanması önerilmektedir; tek grup
kullanılması sıcaklık dengesizliğine yol açabilmektedir.


In [ ]:
def index_yaz(gro, ndx):
    """GRO dosyasindan dogrudan index.ndx yazar."""
    satirlar = open(gro).read().splitlines()
    n = int(satirlar[1])
    gruplar = {'Protein': [], 'Lipid': [], 'Solvent': []}
    for i, s in enumerate(satirlar[2:2+n], start=1):
        rn = s[5:10].strip()
        if rn in LIPIT:
            gruplar['Lipid'].append(i)
        elif rn in COZUCU or rn in IYON:
            gruplar['Solvent'].append(i)
        else:
            gruplar['Protein'].append(i)

    with open(ndx, 'w') as f:
        for ad_g, idx in gruplar.items():
            f.write(f'[ {ad_g} ]\n')
            for k in range(0, len(idx), 15):
                f.write(' '.join(f'{x:>7}' for x in idx[k:k+15]) + '\n')

    sayim = {k: len(v) for k, v in gruplar.items()}
    for g, s in sayim.items():
        print(f'  {g:<10} {s:>10,} parcacik')
    toplam = sum(sayim.values())
    print()
    print(f'  Gruplarin toplami : {toplam:,}')
    print(f'  Sistemdeki toplam : {n:,}')
    print(f'  index.ndx boyutu  : {boyut_str(os.path.getsize(ndx))}')
    print()
    if toplam == n and all(sayim.values()):
        print('DOGRULAMA BASARILI: uc grup sistemi tam ve ortusmeden kapsiyor.')
    elif not all(sayim.values()):
        bos = [g for g, s in sayim.items() if s == 0]
        print(f'UYARI: bos grup(lar) var -> {bos}')
    else:
        print('UYARI: gruplarin toplami sistemle uyusmuyor.')
    return sayim

print('index_yaz() hazir.')


**Aşağıdaki hücre ne yapıyor?** `index.ndx` dosyasını üretip grupların
sistemi eksiksiz kapsadığını doğrulamaktadır.


In [ ]:
index_yaz('sistem.gro', 'index.ndx')


---
## 16. `gmx grompp` ile doğrulama

**Aşağıdaki hücre ne yapıyor?** Koordinat, topoloji ve parametre dosyalarını
birleştirip bir çalıştırma girdisi (`em.tpr`) üretmeye çalışmaktadır. Bu adımda
**simülasyon yürütülmemektedir**; yalnızca dosyaların birbiriyle tutarlılığı
sınanmaktadır.

`.tpr` dosyasının üretilebilmesi, hazırlanan girdinin geçerli olduğunu
göstermektedir. Oturumun hedefi bu doğrulamanın sağlanmasıdır.


In [ ]:
!gmx grompp -f em.mdp -c sistem.gro -p sistem.top -o em.tpr -maxwarn 5 2>&1 | tail -25


**Aşağıdaki hücre ne yapıyor?** Sonucu özetlemektedir.


In [ ]:
if os.path.exists('em.tpr'):
    print(f'BASARILI: em.tpr uretildi ({os.path.getsize("em.tpr"):,} bayt)')
    print('Girdi hazirligi tamamlanmistir.')
else:
    print('em.tpr uretilemedi. Yukaridaki hata iletisi incelenmelidir.')
kaydet(['em.tpr'], D_CIKTI)


**Aşağıdaki hücre ne yapıyor?** Dengeleme aşamasının girdisini de sınamaktadır.
Bu adım `index.ndx` dosyasının ve termostat gruplarının doğru olduğunu
doğrulamaktadır.


In [ ]:
!gmx grompp -f eq.mdp -c sistem.gro -p sistem.top -n index.ndx \
    -o eq_deneme.tpr -maxwarn 10 2>&1 | tail -15

print()
print('BASARILI: termostat gruplari dogru tanimlanmis.'
      if os.path.exists('eq_deneme.tpr') else
      'UYARI: eq.mdp veya index.ndx kontrol edilmeli.')


Sabah oturumunda grafik arayüz ile gerçekleştirilen işlem, bu bölümde komut
satırı araçlarıyla ve kaba-taneli çözünürlükte tamamlanmıştır. Komut satırı
yaklaşımının belirleyici üstünlüğü, işlemin tekrarlanabilir ve raporlanabilir
olmasıdır.


---
## 17. Sistemin analizi ve görselleştirilmesi

Kurulan sistemin doğruluğu görsel ve niceliksel olarak denetlenmelidir.
Beklenen görünüm:

- Lipitler kutunun ortasında iki yaprakçıklı dar bir bant oluşturmalıdır
- Çözücü membranın iki yanında toplanmalı, hidrofobik bölgede bulunmamalıdır
- **Protein membranı dik olarak kat etmeli**, her iki yana taşmalıdır

**Aşağıdaki hücre ne yapıyor?** Sistemi etkileşimli olarak göstermektedir.
Mavi: protein, turuncu: lipit, kırmızı: iyon. Su gizlenmiştir. Görünümü
döndürerek proteinin membran içindeki yönelimini inceleyiniz.


In [ ]:
yapi_goster('sistem.gro', stil='sistem', cozucu_gizle=True)


**Aşağıdaki hücre ne yapıyor?** Kutunun ortasından 2 nm kalınlığında bir dilim
alıp yandan görünümü çizmektedir. 5. bölümdeki yönlendirme sayesinde proteinin
membrana dik durduğu bu çizimde görülmelidir.


In [ ]:
kesit_ciz('sistem.gro', os.path.join(D_GORSEL, '05_sistem_kesit.png'),
          'Adim 5 - AT2R / POPC membran sistemi (yandan kesit)',
          dilim=2.0, eksen=('x', 'z'))


**Aşağıdaki hücre ne yapıyor?** Membran düzlemine dik bakışı çizmektedir.
Proteinin lipitler arasındaki yerleşimi ve çevresinde yeterli lipit bulunup
bulunmadığı denetlenebilir.


In [ ]:
kesit_ciz('sistem.gro', os.path.join(D_GORSEL, '06_sistem_ustten.png'),
          'Adim 6 - membran duzlemi (ustten gorunum)',
          dilim=2.0, eksen=('x', 'y'))


**Aşağıdaki hücre ne yapıyor?** z ekseni boyunca bileşen dağılımını
çizmektedir. Bu, membranın oluşup oluşmadığının niceliksel denetimidir: lipit
eğrisi dar ve tek tepeli, çözücü eğrisi membran bölgesinde sıfıra yakın
olmalıdır.

> Çözücü eğrisindeki periyodik dişli görünüm bir hata değildir; Martini suyu
> düzenli bir ızgaraya yerleştirildiğinden ortaya çıkmaktadır ve dengeleme
> sonrasında kaybolmaktadır.


In [ ]:
z_profili('sistem.gro', os.path.join(D_GORSEL, '07_z_profili.png'),
          'Adim 7 - z ekseni boyunca bilesen dagilimi')


**Aşağıdaki hücre ne yapıyor?** Proteinin son yönelimini ve membran
özelliklerini ölçmektedir. POPC için beklenen değerler: kalınlık yaklaşık
3.8–4.0 nm, lipit başına alan yaklaşık 0.64 nm².


In [ ]:
print('--- Protein yonelimi ---')
aci_son = yonelim_olc('sistem.gro', 'Kurulmus sistem')
print('DEGERLENDIRME: dik yerlesim'
      if aci_son < 35 else 'UYARI: protein membrana yatik olabilir')
print()

res, ad, xyz = koordinat_oku('sistem.gro')
po4 = np.isin(res, list(LIPIT)) & (ad == 'PO4')
z_po4 = xyz[po4, 2]
orta = z_po4.mean()
ust, alt = z_po4[z_po4 > orta], z_po4[z_po4 < orta]
kutu = [float(v) for v in open('sistem.gro').read().splitlines()[-1].split()[:3]]

print('--- Membran ozellikleri ---')
print(f'Kutu boyutlari    : {kutu[0]:.2f} x {kutu[1]:.2f} x {kutu[2]:.2f} nm')
print(f'Lipit sayisi      : ust {len(ust)}, alt {len(alt)}')
print(f'Membran kalinligi : {ust.mean()-alt.mean():.2f} nm   (POPC beklenen ~3.8-4.0)')
print(f'Lipit basina alan : {kutu[0]*kutu[1]/max(len(ust),1):.3f} nm2  (POPC beklenen ~0.64)')
print()
print('Not: bu degerler dengeleme oncesi kurulum degerleridir.')


**Aşağıdaki hücre ne yapıyor?** Sistem bileşimini çizmektedir.


In [ ]:
maskeler = bilesen_maskeleri(res)
sayim = {k: int(v.sum()) for k, v in maskeler.items()}
toplam = sum(sayim.values())

plt.figure(figsize=(7, 4))
plt.bar(list(sayim), list(sayim.values()), color=[RENK[k] for k in sayim])
plt.ylabel('Parcacik sayisi'); plt.yscale('log')
plt.title('Sistem bilesimi')
for i, (k, v) in enumerate(sayim.items()):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
png = os.path.join(D_GORSEL, '08_sistem_bilesimi.png')
plt.savefig(png, dpi=150); plt.show()
print('Kaydedildi:', png)
print()
for k, v in sayim.items():
    print(f'  {k:<8}: {v:>8,}  (%{100*v/toplam:.1f})')


---
## 18. Simülasyon paketinin Drive'a kaydedilmesi

Bu bölümde, GROMACS ile **doğrudan çalıştırılabilecek** eksiksiz bir dosya
kümesi oluşturulmaktadır. Paket kendi kendine yeterlidir: kuvvet alanı
dosyaları dâhil, dışarıdan hiçbir dosyaya bağımlı değildir.

| Dosya | İşlevi |
|---|---|
| `sistem.gro` | Başlangıç koordinatları |
| `sistem.top` | Ana topoloji |
| `molecule_0.itp` | Protein topolojisi (elastik ağ dâhil) |
| `martini_v3.0.0*.itp` | Martini 3 kuvvet alanı parametreleri (4 dosya) |
| `em.mdp`, `eq.mdp`, `md.mdp` | Simülasyon parametreleri |
| `index.ndx` | Termostat grupları |
| `calistir.sh` | Üç aşamayı sırayla yürüten betik |
| `OKUBENI.md` | Kullanım açıklaması |

**Aşağıdaki hücre ne yapıyor?** Çalıştırma betiğini ve açıklama dosyasını
yazmaktadır.


In [ ]:
calistir = '''#!/usr/bin/env bash
#
# Biyofizik 2026 Kursu - Oturum 4
# AT2R (6JOD A zinciri) / POPC membran - Martini 3
#
# Kullanim:  bash calistir.sh
#
set -euo pipefail

# --- 1. Enerji minimizasyonu ---
gmx grompp -f em.mdp -c sistem.gro -p sistem.top -o em.tpr -maxwarn 5
gmx mdrun -deffnm em -v

# --- 2. Dengeleme (100 ps) ---
gmx grompp -f eq.mdp -c em.gro -p sistem.top -n index.ndx -o eq.tpr -maxwarn 5
gmx mdrun -deffnm eq -v

# --- 3. Uretim simulasyonu (1 us) ---
# DIKKAT: bu asama bir GPU uzerinde bile saatler surer.
gmx grompp -f md.mdp -c eq.gro -p sistem.top -n index.ndx -o md.tpr -maxwarn 5
gmx mdrun -deffnm md -v
'''

okubeni = f'''# AT2R / POPC Membran Sistemi - Martini 3

Biyofizik 2026 Kursu, Oturum 4 ciktisi.

## Sistem

- Protein : anjiyotensin II tip-2 reseptoru (PDB 6JOD, A zinciri)
- Yonelim : OPM veritabani (PPM algoritmasi) ile membrana hizalanmis
- Model   : Martini 3 (martini3001), elastik ag (ef=700, el=0.5, eu=0.9)
- Membran : POPC, {len(ust)} + {len(alt)} lipit
- Kutu    : {kutu[0]:.1f} x {kutu[1]:.1f} x {kutu[2]:.1f} nm
- Cozucu  : Martini standart su (W), 0.15 M NaCl
- Toplam  : {toplam:,} parcacik

## Calistirma

```bash
bash calistir.sh
```

Betik uc asamayi sirayla yurutur: enerji minimizasyonu, dengeleme (100 ps),
uretim simulasyonu (1 us).

## Gereksinimler

- GROMACS 2021 veya uzeri
- Uretim asamasi icin GPU siddetle onerilir

## Notlar

- Termostat gruplari `index.ndx` icinde tanimlidir: Protein, Lipid, Solvent
- Membran sistemi oldugu icin barostat `semiisotropic` ayarlanmistir
- Uretim asamasindaki `dt = 0.020` (20 fs) Martini icin tipiktir
- Analiz oncesinde periyodik sinir kosullari duzeltilmelidir:
  `gmx trjconv -s md.tpr -f md.xtc -o md_nojump.xtc -pbc nojump`

## Kaynak

https://github.com/eygpcr/biyofizik2026-martini
'''

open('calistir.sh', 'w').write(calistir)
open('OKUBENI.md', 'w').write(okubeni)
print('calistir.sh ve OKUBENI.md yazildi.')


**Aşağıdaki hücre ne yapıyor?** Tüm dosyaları `simulasyon/` klasörüne
kopyalayıp içeriği listelemektedir. Kuvvet alanı dosyaları da dâhil edildiğinden
paket yaklaşık 17 MB olacaktır.


In [ ]:
prot_itp = sorted(glob.glob('molecule_*.itp'))[0]

paket = ['sistem.gro', 'sistem.top', prot_itp, 'index.ndx',
         'em.mdp', 'eq.mdp', 'md.mdp',
         'calistir.sh', 'OKUBENI.md'] + FF_DOSYALARI

kaydet(paket, D_SIM, sessiz=True)

print('SIMULASYON PAKETI:', D_SIM.replace('/content/drive/MyDrive', "Drive'im"))
print()
toplam_bayt = 0
for d in sorted(os.listdir(D_SIM)):
    b = os.path.getsize(os.path.join(D_SIM, d))
    toplam_bayt += b
    print(f'  {d:<36} {boyut_str(b):>12}')
print(f'\n  Toplam: {toplam_bayt/1e6:.1f} MB')

eksik = [f for f in paket if not os.path.exists(os.path.join(D_SIM, os.path.basename(f)))]
print()
print('DOGRULAMA BASARILI: paket eksiksiz.' if not eksik
      else f'UYARI: eksik dosyalar -> {eksik}')


**Aşağıdaki hücre ne yapıyor?** Paketin gerçekten çalıştırılabilir olduğunu
sınamaktadır: `simulasyon/` klasörüne geçip yalnızca oradaki dosyalarla
`gmx grompp` çalıştırmaktadır. Başarılı olması, paketin dışarıya bağımlı
olmadığını kanıtlamaktadır.


In [ ]:
%cd {D_SIM}
!gmx grompp -f em.mdp -c sistem.gro -p sistem.top -o dogrulama.tpr -maxwarn 5 2>&1 | tail -6
%cd {CALISMA}

dt = os.path.join(D_SIM, 'dogrulama.tpr')
if os.path.exists(dt):
    print()
    print('DOGRULAMA BASARILI: paket kendi basina calistirilabilir.')
    os.remove(dt)
else:
    print()
    print('UYARI: paket eksik olabilir, yukaridaki hata incelenmelidir.')


**Aşağıdaki hücre ne yapıyor?** Tüm Drive klasörünün içeriğini özetlemektedir.


In [ ]:
print('Google Drive icerigi:',
      OTURUM.replace('/content/drive/MyDrive', "Drive'im"))
print()
genel = 0
for alt in ('girdi', 'cikti', 'gorseller', 'simulasyon'):
    yol = os.path.join(OTURUM, alt)
    dosyalar = sorted(os.listdir(yol))
    boyut = sum(os.path.getsize(os.path.join(yol, d)) for d in dosyalar)
    genel += boyut
    print(f'{alt}/ ({len(dosyalar)} dosya, {boyut/1e6:.1f} MB)')
    for d in dosyalar:
        print(f'    {d}')
    print()
print(f'Toplam: {genel/1e6:.1f} MB')


### İsteğe bağlı: paketi bilgisayara indirme

**Aşağıdaki hücre ne yapıyor?** Simülasyon paketini tek bir arşiv hâlinde
indirmektedir. Dosyalar zaten Drive'da saklandığından bu adım gerekli değildir;
yerel bir kopya isteyenler için verilmiştir.


In [ ]:
# import shutil
# from google.colab import files
# arsiv = shutil.make_archive('/content/at2r_martini3', 'zip', D_SIM)
# files.download(arsiv)


---
## Sık karşılaşılan hata iletileri

| Hata iletisi | Nedeni | Çözümü |
|---|---|---|
| Protein membrana yatık duruyor | Yapı membrana yönlendirilmemiş | 5. bölümdeki OPM adımı atlanmamalıdır |
| `DSSPError: Expected record CRYST1 but found ATOM` | `mkdssp` 4.x `CRYST1` kaydı beklemektedir | 7. bölümdeki `-ss` yöntemi kullanılmaktadır |
| `No such moleculetype NA+` | `insane` Martini 2 iyon adları üretmektedir | 12. bölümdeki ad dönüştürme adımı çalıştırılmalıdır |
| `Atomtype X not found` | Kuvvet alanı parametre dosyası eksik | 10. bölümdeki indirmeler denetlenmelidir |
| `number of coordinates does not match topology` | `[ molecules ]` sayıları uyuşmuyor | 13. bölüm yeniden çalıştırılmalıdır |
| `Group Lipid not found` | `index.ndx` eksik veya hatalı | 15. bölüm yeniden çalıştırılmalıdır |
| `System has non-zero total charge` | Yuvarlama kaynaklı; olağandır | `-maxwarn` ile geçilebilir |
| `MessageError: credential propagation was unsuccessful` | Drive bağlama izni verilmedi | 1. bölüm yeniden çalıştırılmalıdır |
| py3Dmol görünümü boş | Tarayıcı JavaScript'i engelliyor olabilir | Hücre yeniden çalıştırılmalı; statik çizimler her durumda üretilmektedir |

---

## Kaynaklar

- [Martini Protein Model — Using Martinize2](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut1.html)
- [Modeling Complex Lipid Membranes — INSANE](https://cgmartini.nl/docs/tutorials/Martini3/LipidsII/)
- [Notes and Limitations](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut4.html)
- [OPM — Orientations of Proteins in Membranes](https://opm.phar.umich.edu/)
- Souza ve ark. (2025). *GōMartini 3.* Nat. Commun. 16. [doi:10.1038/s41467-025-58719-0](https://doi.org/10.1038/s41467-025-58719-0)
- Pedersen ve ark. (2024). *OLIVES.* JCTC. [doi:10.1021/acs.jctc.4c00553](https://doi.org/10.1021/acs.jctc.4c00553)
- Kuvvet alanı dosyaları: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)
